# moeforecast quickstart

This notebook builds a Mixture-of-Experts forecaster step by step on a synthetic hourly demand series. Replace the data cell with your own series; every later cell works unchanged.

In [ ]:
from IPython.display import display

from moeforecast import (
    AdaptiveGating,
    ExpertSelector,
    FeatureBuilder,
    MoEForecaster,
    ProfileClusterer,
    SeriesAnalyzer,
    compare_classifiers,
    feature_relevance,
    prepare_series,
    temporal_train_test_split,
)
from moeforecast.datasets import make_demand_series
from moeforecast.metrics import mae, mape, mase

## 1. Data

`prepare_series` accepts a Series or a DataFrame (with `target` and `time_col`). It sorts the data, checks for duplicates, puts it on a regular grid and reports the gaps.

In [ ]:
raw = make_demand_series(n_days=2 * 364)  # replace with your own data
y, report = prepare_series(raw, return_report=True)
print(report)

train, test = temporal_train_test_split(y, test_size=24 * 28)  # last 4 weeks for testing
y.plot(figsize=(12, 3), linewidth=0.4, title="Hourly demand");

## 2. Which seasonal periods and lags matter?

`SeriesAnalyzer` reads the autocorrelation structure. It suggests periods from the ACF peaks and lags from the significant partial autocorrelations, plus each period and its neighbours. It is computed on the training part only.

In [ ]:
analysis = SeriesAnalyzer().fit(train)
print(analysis.summary())
analysis.plot_acf_pacf()
analysis.plot_seasonal_profiles();

## 3. Features and their relevance

`FeatureBuilder` creates causal features: the value at time *t* uses only data before *t*. Permutation importance on a later hold-out shows which features the model actually relies on.

In [ ]:
features = FeatureBuilder(lags=analysis.suggested_lags_, fourier_periods=analysis.suggested_periods_)
X = features.fit_transform(train)
relevance = feature_relevance(X, train, method="permutation")
relevance.head(10)

## 4. Regimes: day types and week types

Each day (24-hour profile) and each week (168-hour profile) is clustered. `ProfileClusterer.evaluate(train)` exposes candidate metrics and silhouette distributions before fitting; inspect the plots, then choose `n_clusters` explicitly.

In [ ]:
days = ProfileClusterer("D", name="day", k_range=range(2, 11))
days.evaluate(train)
days.plot_scores()

### Compare candidate day-cluster counts

Review the candidate metrics and silhouette shapes before setting the final number of day regimes.

In [ ]:
days.plot_silhouette()  # compare candidate k values before selection

In [ ]:
days.set_params(n_clusters=3).fit(train)
weeks = ProfileClusterer("W", name="week", k_range=range(2, 11))
weeks.evaluate(train)
weeks.plot_scores()

### Compare candidate week-cluster counts

The weekly profiles are evaluated separately; the selected count can differ from the daily one.

In [ ]:
weeks.plot_silhouette()  # compare candidate k values before selection

### Inspect the selected profiles

After reviewing validation, fit each clusterer with the chosen count and inspect the resulting profile groups.

In [ ]:
weeks.set_params(n_clusters=2).fit(train)
print(f"day types: {days.n_clusters_}, week types: {weeks.n_clusters_}")
days.plot_profiles()

In [ ]:
weeks.plot_profiles()

## 5. Can the regimes be predicted from the features?

The gating needs classifiers that predict the regime of each hour. `compare_classifiers` scores candidate classifiers with time-ordered folds.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

classifier_candidates = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "small_forest": RandomForestClassifier(n_estimators=60, min_samples_leaf=5, n_jobs=2, random_state=0),
}
regime_labels = days.labels_for(train.index)
classifier_results = compare_classifiers(X, regime_labels, candidates=classifier_candidates, n_splits=3)
display(classifier_results)

## 6. The Mixture of Experts

`MoEForecaster` fits everything above on the training series. It then selects one expert per (day type, week type) combination, using forward-chaining cross-validation.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = MoEForecaster(
    features=features,
    clusterers=[days, weeks],
    selector=ExpertSelector(
        candidates={"ridge": make_pipeline(StandardScaler(), Ridge())},
        metric="mae",
        n_splits=3,
        min_samples=50,
    ),
    gating=AdaptiveGating(threshold=0.75),
).fit(train)

model.selector_.results_.query("selected")

## 7. Evaluation on the test period

The model was fitted on `train` only. Predicting on the full series gives one-step-ahead forecasts for the test period, each using only the observations before it. `predict_with_metadata` shows which expert was used and whether the gating was hard or soft.

In [ ]:
pred, meta = model.predict_with_metadata(y)
pred, meta = pred[test.index], meta.loc[test.index]

print(f"MAE  {mae(test, pred):.2f}")
print(f"MAPE {mape(test, pred):.2%}")
print(f"MASE {mase(test, pred, train, m=168):.3f}  (< 1 beats the weekly seasonal naive)")
print(meta["mode"].value_counts().to_string())

ax = test.iloc[: 24 * 7].plot(figsize=(12, 3), label="actual")
pred.iloc[: 24 * 7].plot(ax=ax, label="MoE one-step-ahead")
ax.legend();

## 8. Multi-step forecast and saving the model

`forecast` is recursive: each prediction is fed back as the latest observation. `save` and `load` keep the library version alongside the model.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from moeforecast import load, save

week_ahead = model.forecast(train, steps=24 * 7)
print(f"MAE of the recursive 7-day forecast: {mae(test.iloc[: 24 * 7], week_ahead):.2f}")

with TemporaryDirectory() as model_dir:
    model_path = Path(model_dir) / "moe_model.joblib"
    save(model, model_path)
    reloaded = load(model_path)

## 9. Public API coverage and model transparency

The remaining cells exercise the public diagnostic and component-level APIs on the same synthetic example. Internal helpers (names beginning with `_`) are intentionally not part of this walkthrough.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier

from moeforecast import (
    HardGating,
    RegimeClassifier,
    SoftGating,
    default_candidates,
    fill_gaps,
)
from moeforecast.classification import default_classifiers
from moeforecast.gating import joint_probabilities
from moeforecast.metrics import rmse

# Exercise the DataFrame input form and bounded gap filling without
# changing the target used by the fitted model or holdout evaluation.
frame = y.rename_axis("timestamp").rename("demand").reset_index()
prepared_frame = prepare_series(frame, target="demand", time_col="timestamp", freq="h")
assert prepared_frame.equals(y)
gap_example = y.iloc[:100].copy()
gap_example.iloc[20:22] = np.nan
filled_example = fill_gaps(gap_example, max_gap=2, method="time")
print("Gap values remaining:", int(filled_example.iloc[20:22].isna().sum()))

### Analysis, feature ranking, and profile diagnostics

Use both feature-relevance methods, inspect generated feature names, and explore the fitted clustering diagnostics rather than treating the components as a black box.

In [ ]:
analysis.plot_decomposition(periods=[24, 168])
feature_names = features.get_feature_names_out()
print("Number of generated features:", len(feature_names))
mi_ranking = feature_relevance(X, train, method="mutual_info")
permutation_ranking = feature_relevance(X, train, method="permutation")
display(mi_ranking.head(10), permutation_ranking.head(10))

display(days.summary(), weeks.summary())
display(days.profile_matrix(train).head())
display(days.labels_for(train.index).value_counts(dropna=False).sort_index())
weeks.plot_scores()
weeks.plot_profiles()

### Classifier and expert-selection tools

Classifier comparison uses time-ordered folds. The explicit `RegimeClassifier` example shows both class predictions and probabilities; the built-in expert candidates and sample-size policy are also visible.

In [ ]:
built_in_classifiers = default_classifiers(random_state=0)
print("Built-in classifier estimators:", {k: type(v).__name__ for k, v in built_in_classifiers.items()})
display(classifier_results)

valid_rows = X.notna().all(axis=1) & regime_labels.notna()
X_labeled = X.loc[valid_rows]
labels_labeled = regime_labels.loc[valid_rows].astype(int)
regime_classifier = RegimeClassifier(
    estimator=classifier_candidates["logistic"], features=list(X.columns)
).fit(X_labeled, labels_labeled)
classifier_sample = X_labeled.iloc[-5:]
display(
    pd.DataFrame(
        regime_classifier.predict_proba(classifier_sample),
        index=classifier_sample.index,
        columns=regime_classifier.classes_,
    )
)
print("Predicted regimes:", regime_classifier.predict(classifier_sample))

expert_candidates = default_candidates(random_state=0)
selector_demo = ExpertSelector(candidates={"ridge": expert_candidates["ridge"]}, min_samples=50)
print("Default expert estimators:", {k: type(v).__name__ for k, v in expert_candidates.items()})
print("Candidates allowed at n=100:", selector_demo.allowed_candidates(100, list(expert_candidates)))

### Gating strategies and fitted-model inspection

Hard, soft and adaptive routing consume the same regime probabilities. `inspect()` exposes the actual fitted estimators and copied diagnostics for clusters, classifiers, experts and gating.

In [ ]:
gate_sample = X_labeled.iloc[-100:]
gate_probas = [regime_classifier.predict_proba(gate_sample)]
gate_classes = [regime_classifier.classes_]
gate_expert_keys = [(int(label),) for label in regime_classifier.classes_]
joint = joint_probabilities(gate_probas, gate_classes, gate_expert_keys)
print("Joint weights sum to one:", np.allclose(joint.sum(axis=1), 1.0))
for gate in (HardGating(), SoftGating(), AdaptiveGating(threshold=0.75)):
    routed = gate.route(gate_probas, gate_classes, gate_expert_keys)
    print(
        type(gate).__name__,
        "modes:",
        pd.Series(routed.mode).value_counts().to_dict(),
        "normalized weights:",
        np.allclose(routed.weights.sum(axis=1), 1.0),
    )

inspection = model.inspect()
print("Inspection sections:", list(inspection))
display(inspection["clusters"]["day"]["summary"])
display(inspection["classifiers"]["day"]["class_counts"])
display(inspection["expert_selection"])
print("Candidate experts:", {k: type(v).__name__ for k, v in inspection["candidate_experts"].items()})
print("Fitted experts:", {k: type(v).__name__ for k, v in inspection["experts"].items()})
print("Gating estimator:", inspection["gating"])

# The fitted pipeline's other prediction interfaces are aligned by timestamp.
plain_prediction = model.predict(y)
metadata_prediction, metadata_table = model.predict_with_metadata(y)
expert_weight_table = model.expert_weights(y)
assert plain_prediction.equals(metadata_prediction)
display(expert_weight_table.loc[test.index].tail())
print("RMSE:", rmse(test, plain_prediction.loc[test.index]))

reloaded_predictions = reloaded.predict(y)
assert np.allclose(plain_prediction.dropna(), reloaded_predictions.dropna())
print("Persistence check passed: loaded model reproduces predictions.")